# Continue Davis training to 60 total epochs (Kaggle)

This notebook resumes the preserved 30-epoch checkpoints and extends **Davis only** to 60 total epochs. KIBA and Metz stay at their existing 30-epoch checkpoints. The original `cpi_training_checkpoints.zip` is read-only input; this notebook writes a separate `cpi_training_checkpoints_davis60.zip` output.

The saved logs show Davis validation MSE still improving at the 30th epoch (best 0.329098 at epoch 29). KIBA peaked at 0.269671 at epoch 27, and Metz at 0.458601 at epoch 24, so they are not extended here. MSE is a regression error metric, not classification accuracy; lower is better, and MSE scales should not be compared across datasets. Do not use the test set to choose the epoch count.

Attach the original `cpi_kaggle.zip` training dataset and exactly one checkpoint archive: the original `cpi_training_checkpoints.zip`, or a prior `cpi_training_checkpoints_davis*.zip` saved after a continuation. If both archives are attached, set `RESTORE_ZIP` explicitly. Select a Kaggle GPU accelerator and enable Internet for dependency installation. The continuation restores the optimizer/RNG state and uses the same split files and batch size. It does not retrain the other two datasets or overwrite supplied models.

This trains the **2D score model only**. More epochs will not generate protein coordinates or establish ligand binding; the docking/3D workflow is separate. Kaggle GPU runtime and session availability vary, so check the estimate before starting.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

ROOT = Path('/kaggle/working/cpi_project')
INPUT = Path('/kaggle/input')
TARGET_EPOCHS = 60             # Total epochs, not 60 additional epochs.
BATCH_SIZE = 16
RESTORE_ZIP = None             # Auto-detects one attached cpi_training_checkpoints.zip.


def safe_extract(archive, destination, required_prefix=None):
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('Archive contains an unsafe path')
            if required_prefix and not member.filename.startswith(required_prefix):
                raise ValueError('Restore archive must contain only training_runs/')
        z.extractall(destination)


if not (ROOT / 'scripts/train_cpi.py').is_file():
    archives = list(INPUT.rglob('cpi_kaggle.zip'))
    if len(archives) == 1:
        ROOT.mkdir(parents=True, exist_ok=True)
        safe_extract(archives[0], ROOT)
    else:
        sources = [p.parent.parent for p in INPUT.rglob('train_cpi.py')
                   if p.parent.name == 'scripts' and
                   (p.parent.parent / 'model_source/chemprop').is_dir()]
        if len(sources) != 1:
            raise RuntimeError('Attach exactly one CPI training dataset containing cpi_kaggle.zip.')
        shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)

if RESTORE_ZIP is None:
    archives = list(INPUT.rglob('cpi_training_checkpoints*.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected exactly one attached checkpoint ZIP; found {len(archives)}. Attach one archive or set RESTORE_ZIP to its exact path.')
    RESTORE_ZIP = archives[0]
restore_zip = Path(RESTORE_ZIP)
if not restore_zip.is_file():
    raise FileNotFoundError(f'Checkpoint archive not found: {restore_zip}')

training_runs = ROOT / 'training_runs'
if training_runs.exists() and any(training_runs.iterdir()):
    raise RuntimeError('training_runs is not empty. Use a fresh Kaggle working session; the original checkpoint archive is preserved.')
safe_extract(restore_zip, ROOT, 'training_runs/')

for dataset in ('davis', 'kiba', 'metz'):
    run_dir = training_runs / dataset
    manifest_path = run_dir / 'run_manifest.json'
    model_path = run_dir / 'fold_0/model_0/model.pt'
    resume_path = run_dir / 'fold_0/model_0/resume.pt'
    if not all(path.is_file() for path in (manifest_path, model_path, resume_path)):
        raise RuntimeError(f'{dataset} checkpoint archive is missing a manifest, model.pt, or resume.pt.')
    manifest = json.loads(manifest_path.read_text())
    signature = manifest['signature']
    saved_epochs = manifest.get('epochs')
    epochs_compatible = (isinstance(saved_epochs, int) and saved_epochs >= 30
                         if dataset == 'davis' else saved_epochs == 30)
    if not epochs_compatible or signature.get('batch_size') != BATCH_SIZE or signature.get('smoke'):
        raise RuntimeError(f'{dataset} is not a compatible full-data checkpoint for this continuation.')
    for split in ('train', 'val', 'test'):
        source = ROOT / 'processed' / dataset / f'{split}.csv'
        digest = hashlib.sha256(source.read_bytes()).hexdigest()
        if digest != signature['source_sha256'][split]:
            raise RuntimeError(f'{dataset} {split} data does not match the saved checkpoint; refusing to resume.')

print('Restored verified 30-epoch checkpoints from:', restore_zip)
print('Only Davis will be extended; KIBA and Metz remain unchanged.')


In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA runtime:', torch.version.cuda)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing.'
assert torch.cuda.device_count() >= 1, 'At least one CUDA GPU is required.'
x = torch.ones((32, 32), device='cuda:0')
assert (x @ x).sum().item() == 32768
del x
print('GPU 0:', torch.cuda.get_device_name(0))

# Install only additional training packages, preserving Kaggle's supplied PyTorch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(ROOT / 'requirements-kaggle.txt')], check=True)
import importlib.metadata
versions = {name: importlib.metadata.version(name) for name in
            ['torch', 'rdkit', 'numpy', 'pandas', 'scipy', 'scikit-learn',
             'tape-proteins', 'typed-argument-parser', 'tensorboardX', 'lifelines']}
(ROOT / 'kaggle_environment.json').write_text(json.dumps(versions, indent=2))
print(versions)


In [ ]:
LOGS = ROOT / 'kaggle_logs'
LOGS.mkdir(exist_ok=True)


def run_davis_to_target():
    output = ROOT / 'training_runs' / 'davis'
    resume_path = output / 'fold_0/model_0/resume.pt'
    state = torch.load(resume_path, map_location='cpu', weights_only=False)
    completed = int(state['epoch']) + 1
    del state
    print(f'Davis checkpoint has {completed} completed epochs; target is {TARGET_EPOCHS}.', flush=True)
    if completed >= TARGET_EPOCHS:
        print('Davis already meets or exceeds the requested total; no training was started.', flush=True)
        return

    command = [sys.executable, '-u', str(ROOT / 'scripts/train_cpi.py'),
               '--dataset', 'davis', '--device', 'cuda', '--epochs', str(TARGET_EPOCHS),
               '--batch-size', str(BATCH_SIZE), '--threads', '2', '--resume']
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', PYTHONUNBUFFERED='1')
    log_path = LOGS / f'davis_resume_{TARGET_EPOCHS}.log'
    print('Progress log:', log_path, flush=True)
    with log_path.open('a') as log:
        result = subprocess.run(command, cwd=ROOT, env=env,
                                stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        tail = log_path.read_text(errors='replace')[-6000:]
        raise RuntimeError(f'Davis continuation failed. Check the log before retrying.\n{tail}')
    assert (output / 'fold_0/model_0/model.pt').is_file()
    assert resume_path.is_file()
    print('Davis continuation finished.', flush=True)


def save_snapshot():
    archive = Path(f'/kaggle/working/cpi_training_checkpoints_davis{TARGET_EPOCHS}.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in (ROOT / 'training_runs').rglob('*'):
            if path.is_file():
                z.write(path, path.relative_to(ROOT).as_posix())
        progress_log = LOGS / f'davis_resume_{TARGET_EPOCHS}.log'
        if progress_log.is_file():
            z.write(progress_log, f'training_runs/davis/davis_resume_{TARGET_EPOCHS}.log')
    print('Preserved output archive:', archive, flush=True)
    return archive


## Estimate from the saved Kaggle run

The estimate uses the last two measured Davis training and validation epochs from the attached checkpoint archive. It is only a planning estimate: Kaggle GPU allocation, runtime quotas, data loading and final evaluation add variability. The target is 60 total epochs, so the continuation should run about 30 more if the checkpoint is at epoch 30.


In [ ]:
import statistics

davis_output = ROOT / 'training_runs' / 'davis'
state = torch.load(davis_output / 'fold_0/model_0/resume.pt', map_location='cpu', weights_only=False)
completed = int(state['epoch']) + 1
del state
timings = [json.loads(line) for line in
           (davis_output / 'timings.jsonl').read_text().splitlines()]
train = [record['seconds'] for record in timings if record['phase'] == 'training_epoch'][-2:]
validation = [record['seconds'] for record in timings if record['phase'] == 'validation_epoch'][-2:]
if len(train) < 2 or len(validation) < 2:
    raise RuntimeError('The checkpoint archive needs at least two measured train and validation epochs.')
epoch_seconds = statistics.mean(train) + statistics.mean(validation)
remaining_epochs = max(0, TARGET_EPOCHS - completed)
print({
    'completed_epochs': completed,
    'target_total_epochs': TARGET_EPOCHS,
    'remaining_epochs': remaining_epochs,
    'measured_epoch_minutes': round(epoch_seconds / 60, 2),
    'estimated_remaining_hours': round(remaining_epochs * epoch_seconds / 3600, 2),
})


## Continue Davis only

Run this cell after the setup and estimate cells. It resumes the saved optimizer/RNG state and trains toward **60 total epochs**; it does not mean 60 additional epochs. The trainer keeps the model with the best validation MSE, so the final epoch need not be the selected checkpoint. The held-out test set is evaluated by the training program for final reporting, not for deciding whether to continue.

The cell writes a separate checkpoint ZIP even if training raises an error, preserving the latest resumable state when possible. Keep the original ZIP unchanged until the new archive has been downloaded and checked.


In [ ]:
try:
    run_davis_to_target()
    state = torch.load(ROOT / 'training_runs/davis/fold_0/model_0/resume.pt',
                       map_location='cpu', weights_only=False)
    completed = int(state['epoch']) + 1
    del state
    if completed < TARGET_EPOCHS:
        raise RuntimeError(f'Davis stopped at {completed} epochs, below the target of {TARGET_EPOCHS}.')
    print(f'Davis reached {completed} total epochs.', flush=True)
finally:
    save_snapshot()


## Download and verify results

Download `/kaggle/working/cpi_training_checkpoints_davis60.zip` from notebook outputs. It contains the updated Davis checkpoint and unchanged KIBA/Metz checkpoints. Keep the original `cpi_training_checkpoints.zip` as a backup.

Inspect `training_runs/davis/quiet.log` or `kaggle_logs/davis_resume_60.log` for the best validation MSE and selected epoch. `training_runs/davis/fold_0/test_scores.json` contains the final held-out test score; report it once as final evaluation, not as a tuning signal. Check that the KIBA and Metz manifests still say 30 epochs before using the combined archive.

This continuation is not evidence that the metric will improve. Compare validation MSE at the best epoch, and use test MSE/C-index only for a final report. MSE values are dataset-specific and are not percentages or classification accuracy.
